# 🚀 Huấn luyện Mô hình DistilPhoBERT-CRF với Hierarchical Supervised Contrastive Learning trên Google Colab

Notebook này tự động mount **Google Drive**, kiểm tra thư mục dự án `VPhoBertTagger`, nạp bộ dữ liệu **PAP_NER** và thực hiện huấn luyện mô hình **DistilPhoBERT + CRF + Hierarchical SupCon (Weighted Entity-Type Matching)**.

### 📌 Cấu hình huấn luyện tối ưu:
- **Backbone:** `trungbb8/distilphobert` (6 Transformer layers, 92M params, 768-dim)
- **Sequence Tagging:** Linear classifier + `CRF` (Viterbi Decoding)
- **Contrastive Learning:** `Hierarchical SupCon` (`--supcon_mode hierarchical --supcon_alpha 0.3 --lambda_supcon 0.3 --supcon_temp 0.07`)
  - *Strong Positives (cùng BIO & cùng thực thể):* trọng số `w = 1.0`
  - *Weak Positives (cùng thực thể, khác BIO prefix như B-CQ và I-CQ):* trọng số `w = 0.3`
  - *Negatives (khác loại thực thể):* trọng số `w = 0.0`
- **DistilPhoBERT LR:** `2e-5` | **Classifier, CRF & Projection Head LR:** `5e-4` (hoặc `1e-3`)
- **Scheduler:** Linear decay + 10% Warmup | **Batch size:** `128` | **Epochs:** `20` (Early stop: `3`)
- **Output Directory:** `outputs/distilphobert_crf_hierarchical_supcon/`

## Bước 1: Kiểm tra GPU Google Colab
Đảm bảo Colab đang bật GPU (khuyên dùng T4, V100 hoặc A100).

In [1]:
!nvidia-smi

Tue Oct 13 13:57:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Bước 2: Mount Google Drive & Tự động Định vị Thư mục `VPhoBertTagger`

In [2]:
# 1. Mount Google Drive
# Tạo thư  mục tên NER trên drive sau  đó upload toàn thư mục vphoberttagger vào thư mục mới tạo 
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    print('✅ Đã kết nối Google Drive thành công!')
except Exception as e:
    print(f'ℹ️ Ghi chú khi mount Drive: {e}')

# 2. Định vị thư mục VPhoBertTagger trên Google Drive
candidate_paths = [
    '/content/drive/MyDrive/NER/VPhoBertTagger',
]

project_dir = None
for p in candidate_paths:
    if os.path.exists(p) and os.path.exists(os.path.join(p, 'main.py')):
        project_dir = p
        break

if project_dir is None:
    print('🔍 Đang tự động quét tìm thư mục VPhoBertTagger trong Drive...')
    search_roots = ['/content/drive/.shortcut-targets-by-id', '/content/drive/MyDrive']
    for sroot in search_roots:
        if os.path.exists(sroot):
            for root, dirs, files in os.walk(sroot, followlinks=True):
                if 'VPhoBertTagger' in dirs and os.path.exists(os.path.join(root, 'VPhoBertTagger', 'main.py')):
                    project_dir = os.path.join(root, 'VPhoBertTagger')
                    break
                if root.count(os.sep) - sroot.count(os.sep) >= 3:
                    dirs.clear()
            if project_dir:
                break

if project_dir and os.path.exists(project_dir):
    os.chdir(project_dir)
    try:
        get_ipython().run_line_magic('cd', project_dir)
    except Exception:
        pass
    print(f'\n✅ ĐÃ ĐỊNH VỊ VÀ CHUYỂN VÀO THƯ MỤC DỰ ÁN:\n👉 {os.getcwd()}')
else:
    raise FileNotFoundError('❌ Không tìm thấy thư mục VPhoBertTagger trên Google Drive! Vui lòng kiểm tra lại vị trí đặt thư mục trên Drive.')

# Thêm vào PYTHONPATH
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Liệt kê các file trong thư mục dự án
!ls -la


Mounted at /content/drive
✅ Đã kết nối Google Drive thành công!
/content

✅ ĐÃ ĐỊNH VỊ VÀ CHUYỂN VÀO THƯ MỤC DỰ ÁN:
👉 /content/drive/.shortcut-targets-by-id/1P6bQbyn3p5VqMtrbs72FTa_VZwqrzY1W/NER/VPhoBertTagger
total 164
drwx------ 2 root root  4096 Oct 10 00:54  .
drwx------ 3 root root  4096 Oct 10 10:15  ..
-rw------- 1 root root     0 Oct 10 10:56 '=0.7.2'
-rw------- 1 root root     0 Oct 10 10:56 '=4.28.0'
-rw------- 1 root root 18044 Sep  6 13:43  baseline.ipynb
-rw------- 1 root root 10459 Oct  9 09:59  baselineNewLoss.ipynb
-rw------- 1 root root 12896 Sep  6 09:45  baselineV1.ipynb
-rw------- 1 root root 15530 Sep  6 09:33 'baseline V2.ipynb'
-rw------- 1 root root 27158 Oct  8 18:20  CRF_SupCon_Implementation.md
drwx------ 2 root root  4096 Oct 10 00:54  Dataset
drwx------ 2 root root  4096 Oct 10 00:54  docs
drwx------ 2 root root  4096 Oct 10 00:54  .git
-rw------- 1 root root  1070 Sep  6 14:38  .gitignore
drwx------ 2 root root  4096 Oct 10 00:54  img
-rw------- 1 root roo

## Bước 2.1: Kiểm tra & Xác nhận Mã nguồn đã Đồng bộ Hoàn tất (Sync Verification)
Chạy ô bên dưới để tự động kiểm định: **CRF Loss mean**, **Lược bỏ Viterbi lúc train**, **Thứ tự gradient PyTorch**, và **Thang đo hàm mất mát** trước khi bước vào huấn luyện.

In [ ]:
# =====================================================================
# Bước 2.1: Kiểm tra & Xác nhận Toàn diện Mã nguồn Đã Đồng bộ
# (Bao gồm Tối ưu hóa AMP FP16 & Pipeline DataLoader cho GPU A100)
# =====================================================================
import os, sys, inspect, torch

print('=' * 65)
print('🔍 BẮT ĐẦU KIỂM TRA ĐỒNG BỘ MÃ NGUỒN (SYNC VERIFICATION):')
print('=' * 65)

# 1. Kiểm tra mã nguồn trong thư mục dự án:
src_dir = os.path.join(os.getcwd(), 'vphoberttagger')
assert os.path.exists(src_dir), f'❌ Không tìm thấy thư mục {src_dir}!'
print(f'✅ 1. Thư mục mã nguồn cục bộ: {src_dir} (TỒN TẠI)')

# 2. Kiểm tra CRF Loss reduction="mean" trong phobert.py
phobert_file = os.path.join(src_dir, 'models', 'phobert.py')
with open(phobert_file, 'r', encoding='utf-8') as f:
    phobert_code = f.read()
has_crf_mean = "reduction='mean'" in phobert_code
print(f"  2. Cấu hình CRF reduction='mean' (chống nổ 10^35): {'✅ ĐÃ ĐỒNG BỘ' if has_crf_mean else '❌ CHƯA CẬP NHẬT'}")

# 3. Kiểm tra bỏ qua Viterbi Decoding lúc train để tăng tốc
has_viterbi_skip = 'if labels is None or not self.training:' in phobert_code
print(f"  3. Tối ưu tốc độ (bỏ giải mã Viterbi lúc train):   {'✅ ĐÃ ĐỒNG BỘ' if has_viterbi_skip else '❌ CHƯA CẬP NHẬT'}")

# 4. Kiểm tra thứ tự Backward & Gradient Clipping trong trainer.py
trainer_file = os.path.join(src_dir, 'trainer.py')
with open(trainer_file, 'r', encoding='utf-8') as f:
    trainer_code = f.read()
has_backward_before_clip = ('.backward()' in trainer_code and 
                            trainer_code.find('.backward()') < trainer_code.find('torch.nn.utils.clip_grad_norm_'))
print(f"  4. Thứ tự backward() -> clip_grad_norm_() chuẩn:  {'✅ ĐÃ ĐỒNG BỘ' if has_backward_before_clip else '❌ CHƯA CẬP NHẬT'}")

# 5. Kiểm tra tập đánh giá chuyển sang dev_data.txt
has_dev_eval = 'eval_dtype' in trainer_code and "'dev'" in trainer_code
print(f"  5. Sử dụng dev_data.txt để validate (chống leak): {'✅ ĐÃ ĐỒNG BỘ' if has_dev_eval else '❌ CHƯA CẬP NHẬT'}")

# 6. Kiểm tra Kích hoạt Mixed Precision (AMP FP16 GradScaler)
has_amp_scaler = 'GradScaler' in trainer_code and 'autocast' in trainer_code
print(f"  6. Kích hoạt Tensor Cores (AMP FP16 GradScaler):  {'✅ ĐÃ ĐỒNG BỘ' if has_amp_scaler else '❌ CHƯA CẬP NHẬT'}")
# 7. Kiểm tra cách ly CRF trong FP32 chống nổ Loss
has_crf_fp32 = 'logits_fp32' in phobert_code or 'logits.float()' in phobert_code
print(f"  7. Cách ly CRF trong FP32 (chống tràn số):      {'✅ ĐÃ ĐỒNG BỘ' if has_crf_fp32 else '❌ CHƯA CẬP NHẬT'}")

# 7. Kiểm tra Pipeline nạp dữ liệu tốc độ cao (pin_memory=True, num_workers=2)
has_fast_pipeline = 'pin_memory=pin_memory' in trainer_code
print(f"  8. Pipeline dữ liệu (pin_memory + prefetch DMA):   {'✅ ĐÃ ĐỒNG BỘ' if has_fast_pipeline else '❌ CHƯA CẬP NHẬT'}")

# 8. Chạy thử nghiệm Dummy Forward Pass trong RAM
from vphoberttagger.models import PhoBertCrf
from vphoberttagger.constant import LABEL2ID_PAP_NER
from transformers import RobertaConfig

cfg = RobertaConfig(
    vocab_size=500, hidden_size=64, num_hidden_layers=2, num_attention_heads=2,
    intermediate_size=128, max_position_embeddings=256, num_labels=len(LABEL2ID_PAP_NER)
)
setattr(cfg, 'use_supcon', True); setattr(cfg, 'lambda_supcon', 0.3); setattr(cfg, 'supcon_temp', 0.07); setattr(cfg, 'supcon_mode', 'hierarchical'); setattr(cfg, 'supcon_alpha', 0.3); setattr(cfg, 'o_label_id', 0)
m = PhoBertCrf(config=cfg)
m.train()
dummy_x = torch.randint(0, 400, (2, 8))
dummy_mask = torch.ones(2, 8, dtype=torch.long)
dummy_valid = torch.arange(8).unsqueeze(0).repeat(2, 1)
dummy_labels = torch.randint(0, len(LABEL2ID_PAP_NER), (2, 8))
out = m(dummy_x, attention_mask=dummy_mask, valid_ids=dummy_valid, label_masks=dummy_mask, labels=dummy_labels)

is_loss_normal = out.loss_crf.item() < 100.0
is_train_tags_empty = len(out.tags) == 0
print(f"  9. Thang đo Loss thực tế: {out.loss.item():.4f} (CRF: {out.loss_crf.item():.4f}, SupCon: {out.loss_supcon.item():.4f})")
print(f" 10. Thử nghiệm Forward Pass:                       {'✅ CHUẨN XÁC 100%' if (is_loss_normal and is_train_tags_empty) else '❌ BẤT THƯỜNG'}")

all_ok = all([has_crf_mean, has_viterbi_skip, has_backward_before_clip, has_dev_eval, has_amp_scaler, has_crf_fp32, has_fast_pipeline, is_loss_normal, is_train_tags_empty])
print('=' * 65)
if all_ok:
    print('🎉 XÁC NHẬN: TOÀN BỘ MÃ NGUỒN & TỐI ƯU HÓA ĐÃ ĐỒNG BỘ 100%!')
    print('👉 BẠN CÓ THỂ BẮT ĐẦU CHẠY BƯỚC 6 ĐỂ HUẤN LUYỆN SIÊU TỐC TRÊN A100.')
else:
    print('⚠️ CẢNH BÁO: Một số thành phần chưa đồng bộ, vui lòng kiểm tra lại mã nguồn!')
print('=' * 65)


## Bước 3: Cài đặt Thư viện Phụ thuộc (Dependencies)

In [4]:
!pip install -q transformers>=4.28.0 pytorch-crf>=0.7.2 prettytable tensorboardX vncorenlp scikit-learn
!pip install -q -r requirements.txt
print('\n✅ Cài đặt thư viện hoàn tất!')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 81.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 139.5 MB/s eta 0:00:0000:010:01

✅ Cài đặt thư viện hoàn tất!


## Bước 4: Kiểm tra Dữ liệu PAP_NER (`Dataset/`)
Xác nhận sự tồn tại của `train_data.txt`, `dev_data.txt`, `test_data.txt`.

In [6]:
dataset_dir = Path("Dataset")
assert (dataset_dir / "train_data.txt").exists(), "Thiếu file train_data.txt trong Dataset/!"
assert (dataset_dir / "dev_data.txt").exists(), "Thiếu file dev_data.txt trong Dataset/!"
assert (dataset_dir / "test_data.txt").exists(), "Thiếu file test_data.txt trong Dataset/!"

print("✅ Cả 3 file dữ liệu PAP_NER đã sẵn sàng!")
!ls -lh Dataset/*.txt

print("\n--- 5 DÒNG MẪU ĐẦU TIÊN TỪ DEV_DATA.TXT ---")
with open('Dataset/dev_data.txt', 'r', encoding='utf-8') as f:
    for _ in range(5):
        print(f.readline().strip())

✅ Cả 3 file dữ liệu PAP_NER đã sẵn sàng!
-rw------- 1 root root 4.9M Sep  5 16:31 Dataset/dev_data.txt
-rw------- 1 root root 4.8M Sep  5 16:31 Dataset/test_data.txt
-rw------- 1 root root  62M Sep  5 16:31 Dataset/train_data.txt

--- 5 DÒNG MẪU ĐẦU TIÊN TỪ DEV_DATA.TXT ---
Đăng_ký N B-NP O O
thành_lập V B-VP O O
doanh_nghiệp V I-VP B-ĐT O
tư_nhân A O I-ĐT O
do E B-PP O O


## Bước 5: Mở TensorBoard Theo dõi Quá trình Huấn luyện

In [7]:
%load_ext tensorboard
%tensorboard --logdir runs

Launching TensorBoard...

## Bước 6: Huấn luyện Mô hình DistilPhoBERT + CRF + SupCon (Fine-Tuning)
Toàn bộ cấu hình tích hợp Contrastive Learning:
- Model: `trungbb8/distilphobert` + `crf`
- Supervised Contrastive Loss: `--use_supcon --lambda_supcon 0.3 --supcon_temp 0.07`
- DistilPhoBERT LR: `2e-5` | Classifier / CRF / Projection Head LR: `1e-3`
- Output directory: `outputs/distilphobert_crf_hierarchical_supcon`
- Batch Size: `16` | Epochs: `20` | Early stop: `3` epochs

In [8]:
# ============================================================
# Bước 6: Huấn luyện Siêu Tốc trên A100 (AMP FP16 + DataLoader Pinned)
# ============================================================
import os, sys, importlib

# Xóa cache module
for _k in list(sys.modules.keys()):
    if _k == 'vphoberttagger' or _k.startswith('vphoberttagger.'):
        del sys.modules[_k]
importlib.invalidate_caches()

# Đảm bảo sys.path đúng
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Cấu hình tham số huấn luyện tối ưu nhất cho GPU A100
sys.argv = [
    'main.py', 'train',
    '--task', 'pap_ner',
    '--data_dir', './Dataset',
    '--model_name_or_path', 'trungbb8/distilphobert',
    '--model_arch', 'crf',
    '--use_supcon',
    '--lambda_supcon', '0.3',            # Trọng số lambda chuẩn theo CUA-MNER
    '--supcon_temp', '0.07',
    '--supcon_mode', 'hierarchical',      # Kích hoạt Hierarchical Matching
    '--supcon_alpha', '0.3',             # Trọng số weak positive (B/I prefix)
    '--output_dir', 'outputs/distilphobert_crf_hierarchical_supcon',
    '--max_seq_length', '256',
    '--train_batch_size', '128',          # Tăng batch size lên 128 tận dụng 80GB VRAM, giảm 50% số bước lặp
    '--eval_batch_size', '128',
    '--learning_rate', '2e-5',
    '--classifier_learning_rate', '1e-3', # LR tối ưu cho CRF & Classifier
    '--epochs', '20',
    '--warmup_proportion', '0.1',
    '--weight_decay', '0.01',
    '--adam_epsilon', '1e-6',
    '--max_grad_norm', '1.0',
    '--early_stop', '3.0',
    '--num_workers', '2',                 # 2 luồng CPU prefetch song song, xóa nghẽn CPU Colab
    '--fp16',                             # Kích hoạt Tensor Cores (312 TFLOPS) với Mixed Precision
    '--run_test'
]

# Thực thi hàm train trực tiếp
from vphoberttagger.trainer import train
train()


Arguments: Namespace(type='train', task='pap_ner', data_dir='./Dataset', overwrite_data=False, load_weights=None, model_name_or_path='trungbb8/distilphobert', model_arch='crf', use_supcon=True, lambda_supcon=0.3, supcon_temp=0.07, supcon_mode='hierarchical', supcon_alpha=0.3, output_dir='outputs/distilphobert_crf_hierarchical_supcon', max_seq_length=256, train_batch_size=128, eval_batch_size=128, learning_rate=2e-05, classifier_learning_rate=0.001, epochs=20, warmup_proportion=0.1, weight_decay=0.01, adam_epsilon=1e-06, max_grad_norm=1.0, early_stop=3.0, run_test=True, no_cuda=False, seed=42, num_workers=2, fp16=True, save_step=20000, gradient_accumulation_steps=1)
INFO:vphoberttagger.helper:Arguments: Namespace(type='train', task='pap_ner', data_dir='./Dataset', overwrite_data=False, load_weights=None, model_name_or_path='trungbb8/distilphobert', model_arch='crf', use_supcon=True, lambda_supcon=0.3, supcon_temp=0.07, supcon_mode='hierarchical', supcon_alpha=0.3, output_dir='outputs/di

config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  370MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

Using Mixed Precision (AMP FP16): True, Pin Memory: True, Num Workers: 2
INFO:vphoberttagger.helper:Using Mixed Precision (AMP FP16): True, Pin Memory: True, Num Workers: 2

==============================Training epoch 0==============================
INFO:vphoberttagger.helper:
==============================Training epoch 0==============================
[TRAIN-EPOCH 0]: 100%|██████████| 1112/1112 [06:49<00:00,  2.72it/s]
	********************Train Summary********************
INFO:vphoberttagger.helper:	********************Train Summary********************
	Training Lr: 9.963996399639964e-06; Loss: 10.1040; CRF Loss: 8.8076; SupCon Loss: 4.3214; Spend time: 0:06:49.316947
INFO:vphoberttagger.helper:	Training Lr: 9.963996399639964e-06; Loss: 10.1040; CRF Loss: 8.8076; SupCon Loss: 4.3214; Spend time: 0:06:49.316947
[EVAL- Epoch 0]: 100%|██████████| 81/81 [00:16<00:00,  5.02it/s]
	********************Validate Summary********************
INFO:vphoberttagger.helper:	********************Vali

Normalized confusion matrix


## Bước 7: Đánh giá Độc lập trên Tập Test (CoNLL Strict Evaluation)

In [9]:
# Bước 7: Đánh giá trên tập test
import os, sys, importlib
for _k in list(sys.modules.keys()):
    if _k == 'vphoberttagger' or _k.startswith('vphoberttagger.'): del sys.modules[_k]
importlib.invalidate_caches()
if os.getcwd() not in sys.path: sys.path.insert(0, os.getcwd())

sys.argv = [
    'main.py', 'test',
    '--data_dir',    './Dataset',
    '--model_path',  'outputs/distilphobert_crf_hierarchical_supcon/best_model.pt',
]

from vphoberttagger import LOGGER, Trainer
LOGGER.info('Start TEST process...')
Trainer.test()


Start TEST process...
INFO:vphoberttagger.helper:Start TEST process...
Loading checkpoint from 'outputs/distilphobert_crf_hierarchical_supcon/best_model.pt' on device 'cuda'...
INFO:vphoberttagger.helper:Loading checkpoint from 'outputs/distilphobert_crf_hierarchical_supcon/best_model.pt' on device 'cuda'...


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

Running evaluation on test set (10278 samples)...
INFO:vphoberttagger.helper:Running evaluation on test set (10278 samples)...
[EVAL- Epoch 0]: 100%|██████████| 643/643 [01:18<00:00,  8.22it/s]
Processed 243616 tokens with 20979 phrases; Found: 20712 phrases; correct: 20357.
INFO:vphoberttagger.helper:Processed 243616 tokens with 20979 phrases; Found: 20712 phrases; correct: 20357.
Accuracy: 0.9643; (without `O` tag)
INFO:vphoberttagger.helper:Accuracy: 0.9643; (without `O` tag)
Accuracy: 0.9923;  Precision: 0.9829; Recall: 0.9704; F1-score: 0.9766
INFO:vphoberttagger.helper:Accuracy: 0.9923;  Precision: 0.9829; Recall: 0.9704; F1-score: 0.9766
               CQ: Precision: 0.9888; Recall: 0.9684; F1-score: 0.9785  7160
INFO:vphoberttagger.helper:               CQ: Precision: 0.9888; Recall: 0.9684; F1-score: 0.9785  7160
               NG: Precision: 1.0000; Recall: 0.9998; F1-score: 0.9999  4020
INFO:vphoberttagger.helper:               NG: Precision: 1.0000; Recall: 0.9998; F1-score

Normalized confusion matrix


## Bước 8: Tải VnCoreNLP & Dự đoán Thực thể trên Văn bản Mới (Inference)

In [12]:
import os, urllib.request

# 1. Tự động kiểm tra và tải VnCoreNLP nếu chưa có trong thư mục hiện tại
if not os.path.exists("vncorenlp/VnCoreNLP-1.1.1.jar"):
    print("⏳ Đang tải VnCoreNLP (khoảng 10-15 giây)...")
    os.makedirs("vncorenlp/models/wordsegmenter", exist_ok=True)
    base_url = "https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master"
    urllib.request.urlretrieve(f"{base_url}/VnCoreNLP-1.1.1.jar", "vncorenlp/VnCoreNLP-1.1.1.jar")
    urllib.request.urlretrieve(f"{base_url}/models/wordsegmenter/vi-vocab", "vncorenlp/models/wordsegmenter/vi-vocab")
    urllib.request.urlretrieve(f"{base_url}/models/wordsegmenter/wordsegmenter.rdr", "vncorenlp/models/wordsegmenter/wordsegmenter.rdr")
    print("✅ Tải VnCoreNLP thành công!")

# 2. Chạy test dự đoán thực thể với mô hình của bạn
from vphoberttagger.predictor import ViTagger

# Nạp mô hình vừa huấn luyện
tagger = ViTagger(model_path="outputs/distilphobert_crf_hierarchical_supcon/best_model.pt")

# Thử nghiệm câu văn bản pháp luật / hành chính thực tế:
sample_text = "Ủy ban nhân dân tỉnh Thái Bình ban hành Nghị định 123/2020/NĐ-CP ngày 15 tháng 10 năm 2020 quy định về hồ sơ đăng ký doanh nghiệp tư nhân với lệ phí 5 triệu đồng."

entities = tagger(sample_text)
print("\n--- KẾT QUẢ TRÍCH XUẤT THỰC THỂ (LEGAL ENTITIES) ---")
for entity, label in entities:
    print(f"{entity:40} -> [{label}]")


⏳ Đang tải VnCoreNLP (khoảng 10-15 giây)...
✅ Tải VnCoreNLP thành công!
[ViTagger] VnCoreNLP loading ...
[ViTagger] Model loading ...
[ViTagger] All ready!

--- KẾT QUẢ TRÍCH XUẤT THỰC THỂ (LEGAL ENTITIES) ---
Uỷ ban nhân dân tỉnh                     -> [CQ]
Nghị định 123/2020/NĐ-CP                 -> [VBPL]
doanh nghiệp tư nhân                     -> [ĐT]
5 triệu đồng                             -> [SL]


/content/vpho_src/vphoberttagger/predictor.py:95: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  item['valid_ids'] = torch.as_tensor([valid_ids]).to(self.device, dtype=torch.long)
